In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from TICC_solver import TICC
from sklearn.metrics import accuracy_score
from scipy.optimize import linear_sum_assignment

%matplotlib inline

In [ ]:
num_proc = 6     # 並列化（CPUが12コア）

window_size = 1
number_of_clusters = 2
beta = 15
maxIters = 30

In [ ]:
def classify_time_series(df, window_size, number_of_clusters, beta, maxIters, num_proc):
    df.to_csv("data/no_heaer.csv", header=False, index=False, na_rep="NaN")
    
    # TICCインスタンスの作成
    ticc = TICC(window_size=window_size,
                number_of_clusters=number_of_clusters,
                beta=beta,
                maxIters=maxIters,
                threshold=2e-5,
                write_out_file=False,
                prefix_string="output_folder/",
                num_proc=num_proc)
    
    # クラスタリングを実行して cluster_assignment を取得
    cluster_assignment, cluster_MRFs = ticc.fit(input_file="data/no_heaer.csv")
    
    # cluster_assignmentの長さがdfと一致しない場合は先頭にNaNで埋める
    if len(cluster_assignment) != len(df):
        pad_len = len(df) - len(cluster_assignment)
        if pad_len > 0:
            # 先頭にnp.nanで埋める（または-1でもOK）
            cluster_assignment_padded = np.concatenate([np.full(pad_len, np.nan), cluster_assignment])
        else:
            raise ValueError(f"cluster_assignment length ({len(cluster_assignment)}) does not match df length ({len(df)})")
    else:
        cluster_assignment_padded = cluster_assignment

    df["class"] = cluster_assignment_padded

    df1 = df
    df1.to_csv('data/data1_with_class.csv', index=False)
    
    return df1, cluster_assignment_padded

In [ ]:
def plot_variables(df, variables,title=''):
    # color_map = {0: 'blue', 1: 'orange', 2: 'green', 3: 'red', 4: 'purple'}
    color_map = {0: 'blue', 1: 'orange', 4: 'green', 3: 'red', 2: 'purple', 5: 'brown'}

    cluster_series = df['class'].values
    change_points = [0] + list(np.where(cluster_series[1:] != cluster_series[:-1])[0] + 1) + [len(df)]

    for var in variables:
        plt.figure(figsize=(20, 2))

        for start, end in zip(change_points[:-1], change_points[1:]):
            cluster = cluster_series[start]

            # NaN のクラスタはスキップ
            if pd.isna(cluster):
                continue

            color = color_map.get(cluster, 'gray')
            plt.plot(df.index[start:end], df[var].iloc[start:end], color=color)

        plt.xlabel('index')
        plt.ylabel(var)
        plt.title(f'{var} with Cluster Assignments {title}')
        plt.grid()
        plt.show()

In [ ]:
keywords = ['kannno1','kannno2', 'sato1', 'sato2', 'nishi2', 'morita1','morita2','mori1','mori2']

In [ ]:
for kw in keywords:
    # 前処理済みデータの読み込み（精度低い）
    df = pd.read_csv(f"../前処理/data_acc/{kw}_accdata.csv")
    
    # 前処理済みデータの読み込み（精度高い）
    # df = pd.read_csv(f"../前処理/data1/{kw}_data.csv")
    # col = [c for c in df.columns if 'acc' not in c]
    # df = df.drop(columns=col)
    
    # クラスタリング
    classified_df,df_class = classify_time_series(df, window_size, number_of_clusters, beta, maxIters, num_proc)
    df["class"] = df_class
    plot_variables(df, df.columns[:1], title=kw)
    # df.to_csv(f'data_with_class_all/{kw}_data_with_class.csv', index=False)
    df.to_csv(f'data_with_class/{kw}_data_with_class.csv', index=False)

In [ ]:
def clustering_accuracy_per_class(df, true_col='acc_label', cluster_col='class'):
    true_labels = df[true_col].to_numpy()
    cluster_labels = df[cluster_col].to_numpy()

    clusters = np.unique(cluster_labels)
    classes = np.unique(true_labels)

    # --- Hungarian のためのコスト行列 ---
    cost_matrix = np.zeros((len(clusters), len(classes)))
    for i, c in enumerate(clusters):
        for j, t in enumerate(classes):
            cost_matrix[i, j] = -np.sum((cluster_labels == c) & (true_labels == t))

    row_idx, col_idx = linear_sum_assignment(cost_matrix)

    # --- クラスタ → 真のラベルの対応 ---
    mapping = {clusters[r]: classes[c] for r, c in zip(row_idx, col_idx)}

    # --- 予測ラベル ---
    predicted = np.array([mapping[c] for c in cluster_labels])

    # --- 全体精度 ---
    overall_acc = accuracy_score(true_labels, predicted)

    # --- クラス別精度 ---
    per_class_acc = {}
    for cls in classes:
        idx = np.where(true_labels == cls)[0]
        per_class_acc[cls] = accuracy_score(true_labels[idx], predicted[idx])

    return overall_acc, per_class_acc, mapping, predicted

In [ ]:
total_correct = 0
total_samples = 0

major_class_total_correct = 0
major_class_total_samples = 0

minor_class_total_correct = 0
minor_class_total_samples = 0

results = {}

for kw in keywords:
    print(f"\n===== Processing {kw} =====")

    # --- Data読み込み ---
    # df_class = pd.read_csv(f"data_with_class/{kw}_data_with_class.csv")
    df_class = pd.read_csv(f"data_with_class_all/{kw}_data_with_class.csv")
    df_label = pd.read_csv(f"../前処理/label_data/label_{kw}.csv")

    df = pd.concat([df_class, df_label], axis=1)

    # acc_label列にNaNがある行を削除
    df = df.dropna(subset=['acc_label'])


    # --- 精度計算 ---
    accuracy, per_class_acc, mapping, predicted_labels = clustering_accuracy_per_class(df)

    true_labels = df["acc_label"].to_numpy()

    # --- クラス出現数 ---
    unique, counts = np.unique(true_labels, return_counts=True)

    # 多数派 & 少数派
    major_class = unique[np.argmax(counts)]
    minor_class = unique[np.argmin(counts)]

    # 多数派精度
    major_idx = np.where(true_labels == major_class)[0]
    major_correct = np.sum(predicted_labels[major_idx] == true_labels[major_idx])
    major_acc = major_correct / len(major_idx)

    # 少数派精度
    minor_idx = np.where(true_labels == minor_class)[0]
    minor_correct = np.sum(predicted_labels[minor_idx] == true_labels[minor_idx])
    minor_acc = minor_correct / len(minor_idx)

    # 全体精度
    overall_correct = np.sum(predicted_labels == true_labels)
    overall_acc = overall_correct / len(df)

    # ===== 人ごとの結果出力 =====
    print(f"Overall accuracy:     {overall_acc:.4f}")
    print(f"Majority class:       {major_class},  accuracy = {major_acc:.4f}")
    print(f"Minority class:       {minor_class},  accuracy = {minor_acc:.4f}")

    # ===== 全体集計に追加 =====
    total_correct += overall_correct
    total_samples += len(df)

    major_class_total_correct += major_correct
    major_class_total_samples += len(major_idx)

    minor_class_total_correct += minor_correct
    minor_class_total_samples += len(minor_idx)

    # 保存（必要なら使う）
    results[kw] = {
        "overall_accuracy": overall_acc,
        "major_class": major_class,
        "major_accuracy": major_acc,
        "minor_class": minor_class,
        "minor_accuracy": minor_acc,
        "mapping": mapping,
    }

# ================================
#     全員分のまとめ
# ================================
overall_accuracy_all = total_correct / total_samples
major_accuracy_all = major_class_total_correct / major_class_total_samples
minor_accuracy_all = minor_class_total_correct / minor_class_total_samples

print("\n===== Accuracy Summary (All People) =====")
print(f"Overall accuracy (all samples):      {overall_accuracy_all:.4f}")
print(f"Majority-class accuracy (all):       {major_accuracy_all:.4f}")
print(f"Minority-class accuracy (all):       {minor_accuracy_all:.4f}")
